# Ridge Tier-2c: finite-pool features with recent-then-gap base_rates

**Intent:** combine recency (avoid stale critic ecosystem) with gap-matching (embargo-anchor similarity) in the pool definition. Two-stage filter:

1. Take the **50 most recent resolved movies** before target close (recency filter).
2. From those 50, keep the **20 with highest gap_score** `exp(−|target_gap − candidate_gap|/8)` (gap filter).
3. Use those 20 + their gap_scores as weights.
4. Compute weighted per-critic base_rates. Build finite-pool features.

**Hypothesis:** pure A3-gap (tier 2b) picks embargo-similar movies regardless of era, which may include stale critic ecosystems. Pure A1-recency (tier 2) ignores gap. The hybrid captures both.

**LOO:** target excluded from both stages (default_training_slugs filters it in stage 1).

Features: `remaining_base_rate_sum_t2c`, `pool_mass_consumed_t2c`, `observed_top_tier_frac_t2c`.
Compared against all prior Ridge variants.


In [ ]:
import sys
from pathlib import Path

NB_DIR = Path.cwd()
if NB_DIR.name != 'notebooks':
    NB_DIR = NB_DIR / 'notebooks'
if str(NB_DIR) not in sys.path:
    sys.path.insert(0, str(NB_DIR))

import pickle
import time

import numpy as np
import pandas as pd
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import KFold

import _helpers as H

print(f'cohort: {len(H.close_date_map)} movies')


## Noon-shift


In [ ]:
_day_mask = H.reviews['timestamp_confidence'] == 'd'
_n_shifted = int(_day_mask.sum())
H.reviews.loc[_day_mask, 'estimated_timestamp'] = (
    H.reviews.loc[_day_mask, 'estimated_timestamp'] + pd.Timedelta(hours=12)
)
H.first_review_ts = (
    H.reviews[H.reviews['movie_slug'].isin(H.close_date_map)]
    .groupby('movie_slug')['estimated_timestamp'].min()
)
_new_first = H.first_review_ts.to_dict()
H.gaps['first_review_ts'] = H.gaps['slug'].map(_new_first)
H.gaps['gap_days'] = (
    H.gaps['close_ts'] - H.gaps['first_review_ts']
).dt.total_seconds() / 86400
H.gap_lookup = dict(zip(H.gaps['slug'], H.gaps['gap_days']))
print(f'noon-shift: {_n_shifted} day-level reviews')


## Load tier 2b cache (richest starting point)


In [ ]:
T2B_CACHE = H.CACHE_DIR / 'phase1_ridge_tier2b.pkl'
assert T2B_CACHE.exists(), 'run phase1_ridge_tier2b_a3gap.ipynb first'
with open(T2B_CACHE, 'rb') as f:
    df = pickle.load(f)
print(f'loaded {len(df)} rows  ·  columns: {len(df.columns)}')


## Configuration


In [ ]:
SNAP_DAYS_LIST = [5, 4, 3, 2, 1]
BASE_FEATURES = [
    'observed_count', 'first_review_dbc', 'target_gap', 'observed_rate',
    'rate_last_day', 'rate_first_day', 'top_critic_frac',
    'pub_diversity', 'pub_entropy', 'low_activity_frac',
]
TIER1_FEATURES = BASE_FEATURES + [
    'log_observed_count', 'log_rate_last_day', 'sqrt_rate_last_day', 'rate_delta',
]
TIER2C_FEATURES = ['remaining_base_rate_sum_t2c', 'pool_mass_consumed_t2c', 'observed_top_tier_frac_t2c']
ALL_FEATURES = TIER1_FEATURES + TIER2C_FEATURES

STAGE1_POOL_SIZE = 50     # top 50 most recent
STAGE2_POOL_SIZE = 20     # top 20 of those by gap_score
SIGMA_GAP = 8.0
TOP_TIER_N = 30
ALPHA_GRID = [0.01, 0.1, 1.0, 10.0, 100.0, 1000.0]
CV_FOLDS = 5
CV_SEED = 42
CACHE = H.CACHE_DIR / 'phase1_ridge_tier2c.pkl'


## Build recent-then-gap context per target

Stage 1: `default_training_slugs(n=50, before_date=target_close)` — most recent 50 (excludes target).
Stage 2: rank those 50 by `exp(−|gap_diff|/8)`, take top 20.
Weights = gap_score values, normalized so they sum to 20 (matches ship's E2 convention).


In [ ]:
def build_tier2c_context(target_slug):
    close_ts = H.close_date_map[target_slug]
    target_gap = H.gap_lookup.get(target_slug)
    if target_gap is None:
        return None

    # Stage 1: top-50 most recent (excludes target)
    pool50 = H.default_training_slugs(
        H.movies, exclude_slug=target_slug,
        n=STAGE1_POOL_SIZE, before_date=close_ts,
    )
    if len(pool50) < STAGE2_POOL_SIZE:
        return None

    # Stage 2: rank by gap_score, top-20
    scored = []
    for slug in pool50:
        cand_gap = H.gap_lookup.get(slug)
        if cand_gap is None:
            continue
        gap_score = float(np.exp(-abs(target_gap - cand_gap) / SIGMA_GAP))
        scored.append((slug, gap_score))
    scored.sort(key=lambda p: -p[1])
    top20 = scored[:STAGE2_POOL_SIZE]
    if len(top20) < 5:
        return None

    training_slugs = [s for s, _ in top20]
    assert target_slug not in training_slugs, f'leakage: {target_slug} in tier-2c pool'

    raw_weights = np.array([w for _, w in top20], dtype=float)
    n_movies = len(training_slugs)
    total_w = raw_weights.sum()
    if total_w <= 0:
        norm_weights = np.ones_like(raw_weights)
    else:
        norm_weights = raw_weights * (n_movies / total_w)
    slug_weight = dict(zip(training_slugs, norm_weights))

    train = H.reviews[H.reviews['movie_slug'].isin(training_slugs)]
    assert (train['movie_slug'] == target_slug).sum() == 0, 'leakage: target reviews in train'

    base_rate = {}
    for name, group in train.groupby('reviewer_name'):
        movies_seen = group['movie_slug'].unique()
        weighted_ratio = float(sum(slug_weight[s] for s in movies_seen) / n_movies)
        base_rate[name] = weighted_ratio

    total_sum = float(sum(base_rate.values()))
    sorted_critics = sorted(base_rate.items(), key=lambda p: -p[1])
    top_tier = set(c for c, _ in sorted_critics[:TOP_TIER_N])

    return {
        'training_slugs': training_slugs,
        'base_rate': base_rate,
        'total_sum': total_sum,
        'top_tier': top_tier,
        'weights': slug_weight,
        'gap_distances': [abs(target_gap - H.gap_lookup[s]) for s in training_slugs],
    }


t2c_cache = {}
start = time.time()
for slug in sorted(H.close_date_map):
    ctx = build_tier2c_context(slug)
    if ctx is not None:
        t2c_cache[slug] = ctx
print(f'built tier-2c pool for {len(t2c_cache)} targets  ·  {time.time()-start:.1f}s')

# Diagnostic: typical gap-distance distribution in tier-2c training
all_gap_dists = []
for ctx in t2c_cache.values():
    all_gap_dists.extend(ctx['gap_distances'])
arr = np.array(all_gap_dists)
print(f'\ntier-2c gap distances (|target_gap − train_gap|) across all target-train pairs:')
print(f'  n={len(arr)}  median={np.median(arr):.2f}d  p75={np.quantile(arr, 0.75):.2f}d  p95={np.quantile(arr, 0.95):.2f}d  max={arr.max():.2f}d')


## Diagnostic: how different is tier-2c pool from A1 and A3 pools?


In [ ]:
def build_a1_pool(target_slug):
    close_ts = H.close_date_map[target_slug]
    return set(H.default_training_slugs(
        H.movies, exclude_slug=target_slug, n=STAGE2_POOL_SIZE, before_date=close_ts,
    ))


def build_a3_pool(target_slug):
    target_gap = H.gap_lookup.get(target_slug)
    if target_gap is None:
        return set()
    scores = H.combined_score_with_scores(
        target=target_slug, target_gap=target_gap,
        target_critics=set(), target_window_days=1.0,
        k=STAGE2_POOL_SIZE, alpha=1.0, sigma_gap=SIGMA_GAP,
    )
    return set(scores.keys())


t2c_vs_a1_jacc = []
t2c_vs_a3_jacc = []
for slug in list(t2c_cache.keys())[:50]:
    t2c_set = set(t2c_cache[slug]['training_slugs'])
    a1_set = build_a1_pool(slug)
    a3_set = build_a3_pool(slug)
    if t2c_set and a1_set:
        t2c_vs_a1_jacc.append(len(t2c_set & a1_set) / len(t2c_set | a1_set))
    if t2c_set and a3_set:
        t2c_vs_a3_jacc.append(len(t2c_set & a3_set) / len(t2c_set | a3_set))

print(f't2c pool vs A1-recency (n={len(t2c_vs_a1_jacc)}):')
print(f'  Jaccard median={np.median(t2c_vs_a1_jacc):.2f}  mean={np.mean(t2c_vs_a1_jacc):.2f}')
print(f't2c pool vs A3-gap (n={len(t2c_vs_a3_jacc)}):')
print(f'  Jaccard median={np.median(t2c_vs_a3_jacc):.2f}  mean={np.mean(t2c_vs_a3_jacc):.2f}')


## Compute tier-2c finite-pool features


In [ ]:
def compute_pool_features_t2c(target_slug, snap_days):
    ctx = t2c_cache.get(target_slug)
    if ctx is None:
        return None
    close_ts = H.close_date_map[target_slug]
    midnight_utc = close_ts.floor('D')
    snap_time = midnight_utc - pd.Timedelta(days=snap_days)
    state = H.snapshot_state(target_slug, snap_time)
    if state is None:
        return None
    observed = state['observed_critics']

    base_rate = ctx['base_rate']
    total_sum = ctx['total_sum']
    top_tier = ctx['top_tier']

    obs_br_sum = float(sum(base_rate.get(c, 0.0) for c in observed))
    remaining_br_sum = max(total_sum - obs_br_sum, 0.0)
    pool_mass_consumed = (obs_br_sum / total_sum) if total_sum > 0 else 0.0
    top_observed = len(observed & top_tier)
    observed_top_tier_frac = top_observed / TOP_TIER_N

    return {
        'remaining_base_rate_sum_t2c': remaining_br_sum,
        'pool_mass_consumed_t2c': pool_mass_consumed,
        'observed_top_tier_frac_t2c': observed_top_tier_frac,
    }


for feat in TIER2C_FEATURES:
    df[feat] = np.nan

start = time.time()
for idx, row in df.iterrows():
    feats = compute_pool_features_t2c(row['target_slug'], int(row['snap_days']))
    if feats is None:
        continue
    for k, v in feats.items():
        df.at[idx, k] = v
print(f'computed tier-2c pool features in {time.time()-start:.1f}s')

print('\nfeature correlations (t2c vs a1 vs a3) at T-3d:')
corr_feats = [
    'remaining_base_rate_sum', 'remaining_base_rate_sum_a3', 'remaining_base_rate_sum_t2c',
    'pool_mass_consumed', 'pool_mass_consumed_a3', 'pool_mass_consumed_t2c',
    'observed_top_tier_frac', 'observed_top_tier_frac_a3', 'observed_top_tier_frac_t2c',
]
print(df[df['snap_days'] == 3][corr_feats].corr().round(3).to_string())


## α CV + LOO predict


In [ ]:
def select_alpha(X, y):
    kf = KFold(n_splits=CV_FOLDS, shuffle=True, random_state=CV_SEED)
    best_alpha, best_mae = None, np.inf
    for alpha in ALPHA_GRID:
        fold_errs = []
        for train_idx, test_idx in kf.split(X):
            pipe = Pipeline([('scaler', StandardScaler()), ('ridge', Ridge(alpha=alpha))])
            pipe.fit(X[train_idx], y[train_idx])
            preds = pipe.predict(X[test_idx])
            fold_errs.extend(np.abs(preds - y[test_idx]).tolist())
        mae = float(np.mean(fold_errs))
        if mae < best_mae:
            best_mae, best_alpha = mae, alpha
    return best_alpha


def loo_predict(X, y, alpha):
    preds = np.zeros(len(X))
    for i in range(len(X)):
        mask = np.ones(len(X), dtype=bool)
        mask[i] = False
        pipe = Pipeline([('scaler', StandardScaler()), ('ridge', Ridge(alpha=alpha))])
        pipe.fit(X[mask], y[mask])
        preds[i] = pipe.predict(X[i:i+1])[0]
    return preds


snap_alpha_t2c = {}
df['ridge_t2c_pred'] = np.nan
for snap_days in SNAP_DAYS_LIST:
    sub = df[df['snap_days'] == snap_days].dropna(subset=ALL_FEATURES + ['actual'])
    if len(sub) < CV_FOLDS * 2:
        continue
    X = sub[ALL_FEATURES].values
    y = sub['actual'].values.astype(float)
    best_alpha = select_alpha(X, y)
    snap_alpha_t2c[snap_days] = best_alpha
    preds = loo_predict(X, y, best_alpha)
    df.loc[sub.index, 'ridge_t2c_pred'] = preds
    print(f'  T-{snap_days}d (n={len(sub)}): α*={best_alpha}')

with open(CACHE, 'wb') as f:
    pickle.dump(df, f)
print(f'saved {CACHE}')


## Per-variant cohort summary (7-way)


In [ ]:
def metrics(sub, pred_col):
    s = sub.dropna(subset=[pred_col])
    if len(s) == 0:
        return None
    err = s[pred_col].values - s['actual'].values
    return {
        'n': len(s), 'MAE': float(np.abs(err).mean()),
        'me': float(err.mean()),
        'p90_abs_err': float(np.quantile(np.abs(err), 0.9)),
    }


VARIANTS = [
    ('library',    'lib_pred'),
    ('ship',       'ship_pred'),
    ('ridge_orig', 'ridge_pred'),
    ('ridge_t1',   'ridge_t1_pred'),
    ('ridge_t2',   'ridge_t2_pred'),
    ('ridge_t2b',  'ridge_t2b_pred'),
    ('ridge_t2c',  'ridge_t2c_pred'),
]

print('=== cohort summary ===\n')
for snap_days in SNAP_DAYS_LIST:
    sub = df[df['snap_days'] == snap_days]
    print(f'T-{snap_days}d')
    for name, col in VARIANTS:
        m = metrics(sub, col)
        if m:
            print(f'  {name:12s}  n={m["n"]:3d}  MAE={m["MAE"]:6.2f}  '
                  f'me={m["me"]:+6.2f}  p90|e|={m["p90_abs_err"]:6.2f}')
    print()


## h/m subset


In [ ]:
HM = ['the_drama', 'the_super_mario_galaxy_movie', 'forbidden_fruits_2026',
      'they_will_kill_you', 'you_me_and_tuscany']
hm_df = df[df['target_slug'].isin(HM)]

print('=== h/m subset ===\n')
for snap_days in SNAP_DAYS_LIST:
    sub = hm_df[hm_df['snap_days'] == snap_days]
    if sub.empty:
        continue
    print(f'T-{snap_days}d (n={len(sub)})')
    for name, col in VARIANTS:
        m = metrics(sub, col)
        if m:
            print(f'  {name:12s}  MAE={m["MAE"]:6.2f}  me={m["me"]:+6.2f}')
    print()


## Paired bootstrap (tier 2c vs all prior variants)


In [ ]:
COL_MAP = {name: col for name, col in VARIANTS}
PAIRS = [
    ('ridge_orig', 'ridge_t2c'),
    ('ridge_t1',   'ridge_t2c'),
    ('ridge_t2',   'ridge_t2c'),
    ('ridge_t2b',  'ridge_t2c'),
]

print('=== paired bootstrap ΔMAE (A − B, +ve → B wins) ===\n')
print(f'{"snap":<6}{"A":<12}{"B":<12}{"Δ":>10}{"CI95_lo":>10}{"CI95_hi":>10}{"Δ %":>8}{"n":>6}  result')
for snap_days in SNAP_DAYS_LIST:
    snap_df = df[df['snap_days'] == snap_days]
    for a, b in PAIRS:
        a_col, b_col = COL_MAP[a], COL_MAP[b]
        paired = snap_df.dropna(subset=[a_col, b_col, 'actual'])
        if len(paired) < 5:
            continue
        a_abs = np.abs(paired[a_col].values - paired['actual'].values)
        b_abs = np.abs(paired[b_col].values - paired['actual'].values)
        deltas = a_abs - b_abs
        point, lo, hi = H.bootstrap_mae_delta(deltas, n_boot=1000)
        pct = 100 * point / a_abs.mean() if a_abs.mean() else float('nan')
        if lo > 0:
            result = f'{b:>12} wins'
        elif hi < 0:
            result = f'{a:>12} wins'
        else:
            result = '    ns'
        print(f'T-{snap_days}d  {a:<12}{b:<12}{point:>+10.3f}{lo:>+10.3f}{hi:>+10.3f}{pct:>+8.2f}{len(paired):>6}  {result}')
    print()


## Coefficients


In [ ]:
print('=== tier-2c coefficients (standardized) ===\n')
for snap_days in SNAP_DAYS_LIST:
    alpha = snap_alpha_t2c.get(snap_days)
    if alpha is None:
        continue
    sub = df[df['snap_days'] == snap_days].dropna(subset=ALL_FEATURES + ['actual'])
    if len(sub) < 10:
        continue
    X = sub[ALL_FEATURES].values
    y = sub['actual'].values.astype(float)
    pipe = Pipeline([('scaler', StandardScaler()), ('ridge', Ridge(alpha=alpha))])
    pipe.fit(X, y)
    coefs = pipe.named_steps['ridge'].coef_
    intercept = pipe.named_steps['ridge'].intercept_
    pairs = sorted(zip(ALL_FEATURES, coefs), key=lambda p: -abs(p[1]))
    print(f'T-{snap_days}d  α={alpha}  intercept={intercept:.2f}')
    for feat, c in pairs:
        marker = '  ← pool(t2c)' if feat in TIER2C_FEATURES else ''
        print(f'  {feat:30s}  {c:+7.2f}{marker}')
    print()


## Plot


In [ ]:
import matplotlib.pyplot as plt

summary_rows = []
for snap_days in SNAP_DAYS_LIST:
    sub = df[df['snap_days'] == snap_days]
    for name, col in VARIANTS:
        m = metrics(sub, col)
        if m:
            summary_rows.append({'snap_days': snap_days, 'variant': name, **m})
summary = pd.DataFrame(summary_rows)

fig, ax = plt.subplots(1, 1, figsize=(12, 4.5))
colors = {'library': 'tab:gray', 'ship': 'tab:red',
          'ridge_orig': 'tab:blue', 'ridge_t1': 'tab:green',
          'ridge_t2': 'tab:purple', 'ridge_t2b': 'tab:orange',
          'ridge_t2c': 'tab:brown'}
markers = {'library': 's', 'ship': 'o', 'ridge_orig': '^',
           'ridge_t1': 'D', 'ridge_t2': '*', 'ridge_t2b': 'P', 'ridge_t2c': 'X'}
for name, _ in VARIANTS:
    sub = summary[summary['variant'] == name].sort_values('snap_days', ascending=False)
    ax.plot(sub['snap_days'], sub['MAE'], '-',
            marker=markers[name], color=colors[name], label=name, markersize=9)
ax.invert_xaxis()
ax.set_xlabel('snap days before close')
ax.set_ylabel('MAE (reviews)')
ax.set_title('Phase-1 MAE — tier-2c (top-50 recent, top-20 gap) vs all')
ax.legend(loc='upper right')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()


## Observations

*(fill in after run)*
